# Generate Item Embeddings (Kaggle GPU)

**Подготовка:**
1. Загрузи `Pet_Supplies_items_merged.parquet` как Kaggle Dataset (New Dataset → Upload file)
2. Добавь его в этот ноутбук: Add data → ваш датасет
3. Убедись что включён GPU: Settings → Accelerator → GPU T4 x1
4. Запусти все ячейки
5. Скачай `/kaggle/working/Pet_Supplies_items_with_embeddings.parquet`

In [ ]:
# Установка зависимостей (нужна только transformers, torch уже есть на Kaggle)
!pip install -q transformers accelerate polars pyarrow

In [ ]:
import os
import gc
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from tqdm.auto import tqdm

# === Настройки ===
CATEGORY        = "Pet_Supplies"
MODEL_NAME      = "Qwen/Qwen3-Embedding-0.6B"  # ~1.2 GB VRAM, подходит для T4
BATCH_SIZE      = 32    # 32 безопасно для T4 15GB; увеличь до 64 если памяти хватает
MAX_LENGTH      = 512   # item_context median ~300 токенов, max ~1000
TARGET_DIM      = 1024  # размерность эмбеддинга
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
N_GPUS          = torch.cuda.device_count() if DEVICE == "cuda" else 0

# Пути (Kaggle: датасет монтируется в /kaggle/input/<dataset-name>/)
# Если название датасета другое — измени путь ниже
INPUT_PATHS = [
    f"/kaggle/input/pet-supplies-merged/{CATEGORY}_items_merged.parquet",
    f"/kaggle/input/pet-supplies/{CATEGORY}_items_merged.parquet",
    f"/kaggle/input/{CATEGORY.lower().replace('_', '-')}/{CATEGORY}_items_merged.parquet",
]
OUTPUT_PATH = f"/kaggle/working/{CATEGORY}_items_with_embeddings.parquet"

# Найти файл
INPUT_PATH = None
for p in INPUT_PATHS:
    if Path(p).exists():
        INPUT_PATH = p
        break

if INPUT_PATH is None:
    # Найти автоматически
    import glob
    found = glob.glob(f"/kaggle/input/**/*merged*.parquet", recursive=True)
    if found:
        INPUT_PATH = found[0]
        print(f"Auto-found: {INPUT_PATH}")
    else:
        raise FileNotFoundError(
            "Не найден файл items_merged.parquet.\n"
            f"Проверь путь в /kaggle/input/: {os.listdir('/kaggle/input/')}"
        )

print(f"Device : {DEVICE}")
print(f"GPUs   : {N_GPUS}")
print(f"Input  : {INPUT_PATH}")
print(f"Output : {OUTPUT_PATH}")
if DEVICE == "cuda":
    for i in range(N_GPUS):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  {props.total_memory / 1e9:.1f} GB")

In [ ]:
# Загружаем данные
df = pl.read_parquet(INPUT_PATH)
print(f"Loaded: {df.shape}")
print(f"Columns: {df.columns}")

# Проверка обязательных колонок
assert "parent_asin" in df.columns, "Нет колонки parent_asin"
assert "item_context" in df.columns, "Нет колонки item_context"

# Проверка null в item_context
n_null = df["item_context"].null_count()
if n_null > 0:
    print(f"WARNING: {n_null} null в item_context — заменяем пустой строкой")
    df = df.with_columns(
        pl.col("item_context").fill_null(pl.col("title"))
    )

texts = df["item_context"].to_list()
asins = df["parent_asin"].to_list()

print(f"\nitem_context stats:")
ctx_len = df["item_context"].str.len_chars()
print(f"  min={ctx_len.min()}, median={ctx_len.median():.0f}, max={ctx_len.max()} chars")

In [ ]:
# Загружаем по одной копии модели на каждый GPU
# DataParallel не подходит для decoder-моделей (gather переполняет GPU 0)
# Вместо этого: две независимые копии + ThreadPoolExecutor
print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

gpu_models = {}
for gpu_id in range(max(1, N_GPUS)):
    device = f"cuda:{gpu_id}" if DEVICE == "cuda" else "cpu"
    m = AutoModel.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
    m = m.to(device).eval()
    gpu_models[gpu_id] = m
    if DEVICE == "cuda":
        mem = torch.cuda.memory_allocated(gpu_id) / 1e9
        print(f"  GPU {gpu_id}: model loaded, VRAM used = {mem:.2f} GB")

print(f"Models ready on {len(gpu_models)} device(s).")

In [ ]:
import threading

def last_token_pool(last_hidden_states: torch.Tensor,
                    attention_mask: torch.Tensor) -> torch.Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    sequence_lengths = attention_mask.sum(dim=1) - 1
    batch_size = last_hidden_states.shape[0]
    return last_hidden_states[
        torch.arange(batch_size, device=last_hidden_states.device),
        sequence_lengths,
    ]


def process_chunk(chunk_texts, gpu_id, result_array, offset, pbar, pbar_lock):
    """Обрабатывает свою половину данных независимо на одном GPU.

    Каждый поток ведёт собственный цикл — нет конкуренции за очередь задач,
    нет gather между GPU, минимальная нагрузка на GIL.
    """
    device = f"cuda:{gpu_id}" if DEVICE == "cuda" else "cpu"
    model = gpu_models[gpu_id]

    for i in range(0, len(chunk_texts), BATCH_SIZE):
        batch = chunk_texts[i : i + BATCH_SIZE]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = model(**encoded)

        emb = last_token_pool(outputs.last_hidden_state, encoded["attention_mask"])
        if TARGET_DIM < emb.shape[1]:
            emb = emb[:, :TARGET_DIM]
        emb = F.normalize(emb, p=2, dim=1).cpu().float().numpy()

        result_array[offset + i : offset + i + len(batch)] = emb

        # Освобождаем кеш после каждого батча
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

        with pbar_lock:
            pbar.update(1)


# Делим данные ровно пополам между GPU
N = len(texts)
all_embeddings = np.zeros((N, TARGET_DIM), dtype=np.float32)
n_workers = len(gpu_models)
mid = N // n_workers  # граница раздела

chunks = []
for gpu_id in range(n_workers):
    start = gpu_id * mid
    end = N if gpu_id == n_workers - 1 else (gpu_id + 1) * mid
    chunks.append((texts[start:end], gpu_id, start))

total_batches = sum(
    (len(c[0]) + BATCH_SIZE - 1) // BATCH_SIZE for c in chunks
)

print(f"Generating embeddings for {N:,} items")
print(f"  batch_size={BATCH_SIZE}, total_batches={total_batches}, GPUs={n_workers}")
for gpu_id, (chunk_texts, gid, offset) in enumerate(chunks):
    print(f"  GPU {gid}: items {offset}..{offset + len(chunk_texts) - 1} ({len(chunk_texts):,} items)")

pbar = tqdm(total=total_batches, desc="Batches")
pbar_lock = threading.Lock()

threads = [
    threading.Thread(
        target=process_chunk,
        args=(chunk_texts, gpu_id, all_embeddings, offset, pbar, pbar_lock),
        daemon=True,
    )
    for chunk_texts, gpu_id, offset in chunks
]

for t in threads:
    t.start()
for t in threads:
    t.join()

pbar.close()

print(f"\nDone. Embeddings shape: {all_embeddings.shape}")
norms = np.linalg.norm(all_embeddings, axis=1)
print(f"L2 norms — mean: {norms.mean():.4f}, std: {norms.std():.4f}  (должны быть ~1.0)")

In [ ]:
# Сохраняем: только parent_asin + embedding (остальные поля уже есть в merged)
embeddings_list = all_embeddings.tolist()

out_df = pl.DataFrame({
    "parent_asin": asins,
    "embedding": embeddings_list,
})

# Проверка
assert out_df.height == N, f"Row count mismatch: {out_df.height} != {N}"
assert out_df["parent_asin"].n_unique() == N, "Дубликаты parent_asin!"

out_df.write_parquet(OUTPUT_PATH)

size_mb = Path(OUTPUT_PATH).stat().st_size / 1024 / 1024
print(f"Saved: {OUTPUT_PATH}")
print(f"Size : {size_mb:.1f} MB")
print(f"Rows : {out_df.height:,}")
print(f"Schema: {out_df.schema}")
print(out_df.head(3))

In [ ]:
# Быстрая проверка качества: cosine similarity между похожими товарами
# (просто смотрим что ближайшие соседи среди первых 1000 выглядят разумно)
sample_size = min(1000, N)
sample_emb = all_embeddings[:sample_size]  # [1000, 1024]
sim_matrix = sample_emb @ sample_emb.T     # [1000, 1000]

# Для каждого товара найдём ближайшего соседа (кроме самого себя)
np.fill_diagonal(sim_matrix, -1)
top1_indices = sim_matrix.argmax(axis=1)
top1_sims = sim_matrix[np.arange(sample_size), top1_indices]

print(f"Nearest-neighbor similarity (first {sample_size} items):")
print(f"  mean={top1_sims.mean():.3f}, min={top1_sims.min():.3f}, max={top1_sims.max():.3f}")
print()

# Показываем 3 примера
sample_df = df.head(sample_size)
for i in [0, 50, 100]:
    j = int(top1_indices[i])   # numpy.int64 → Python int для Polars
    print(f"  Item  : {sample_df['title'][i][:70]}")
    print(f"  Nearest: {sample_df['title'][j][:70]}  (sim={top1_sims[i]:.3f})")
    print()